In [2]:
# 1. Install the Groq library silently
!pip install groq -q

import os
from google.colab import userdata
from groq import Groq

# 2. Securely load the API key from Colab Secrets
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
except userdata.SecretNotFoundError:
    print("Error: Please add GROQ_API_KEY to your Colab Secrets (the key icon on the left).")

# 3. Initialize the Groq client
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# 4. Define the precise few-shot prompt
SYSTEM_PROMPT = """
You are the routing engine for a mental health support chatbot.
Your ONLY job is to classify the user's input into exactly ONE of the following five categories:

1. greeting: The user is saying hello or introducing themselves.
2. goodbye: The user is ending the conversation.
3. gratitude: The user is saying thank you or expressing appreciation.
4. asking_mental_health_question: The user is discussing their feelings, asking for advice, discussing anxiety, depression, stress, or seeking crisis support.
5. out_of_scope: The user is asking about the weather, coding, math, general trivia, or anything completely unrelated to mental health.

Respond with ONLY the exact category name. Do not include punctuation, explanations, or conversational text.

Examples:
User: "Hi there, how are you?"
greeting

User: "Thanks for listening to me."
gratitude

User: "I can't stop crying and my chest hurts from anxiety."
asking_mental_health_question

User: "Can you write a Python script for me?"
out_of_scope

User: "See you later."
goodbye
"""

# 5. The Routing Function
def classify_intent(user_input: str) -> str:
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_input}
            ],
            model="llama-3.3-70b-versatile",
            temperature=0.0,
            max_tokens=10,
        )

        intent = chat_completion.choices[0].message.content.strip()

        valid_intents = ["greeting", "goodbye", "gratitude", "asking_mental_health_question", "out_of_scope"]
        if intent not in valid_intents:
            print(f"Warning: Unexpected LLM output: '{intent}'. Defaulting to 'out_of_scope'.")
            return "out_of_scope"

        return intent

    except Exception as e:
        print(f"Groq API Error: {e}")
        return "asking_mental_health_question"

# 6. Run the Test
test_inputs = [
    "Good morning!",
    "I'm feeling really overwhelmed with my exams coming up.",
    "What is the capital of France?",
    "I appreciate your help earlier.",
    "Talk to you tomorrow."
]

print("Testing Intent Classifier...\n")
for text in test_inputs:
    intent = classify_intent(text)
    print(f"Input:  \"{text}\"")
    print(f"Intent: {intent}\n")

Testing Intent Classifier...

Input:  "Good morning!"
Intent: greeting

Input:  "I'm feeling really overwhelmed with my exams coming up."
Intent: asking_mental_health_question

Input:  "What is the capital of France?"
Intent: out_of_scope

Input:  "I appreciate your help earlier."
Intent: gratitude

Input:  "Talk to you tomorrow."
Intent: goodbye

